# 05. Week 3 - Dataset Charter and Processed Dataset V1

    Objective: prove that the anime recommender has a valid domain, valid sources, a reproducible first dataset build, and a schema that supports representation, clustering, recommendation, and graph analysis.

## Project Proposal

    *Domain.* Anime discovery and recommendation.

    *Problem statement.* Anime catalogs are large, sparse, and fragmented across platforms. A viewer may know a few shows they like but still struggle to discover similar titles, hidden related works, or thematic clusters. This project builds a reproducible anime catalog that combines MyAnimeList metadata, AniDB tags, MAL recommendation edges, MAL relation edges, and optional user ratings.

    *Product question.* Given an anime or a viewer preference profile, which anime are similar, which latent segment do they belong to, and which titles should be recommended next?

    *Course fit.* The dataset supports a catalog layer, feature layer, interaction layer, graph layer, and reproducible pipeline. It is not a single supervised model project.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
BUILD_DIR = DATA_DIR / "build"
PLOT_DIR = BASE_DIR / "artifacts" / "plots"

ANIME_PATH = PROCESSED_DIR / "anime_dataset.csv"
RATINGS_PATH = PROCESSED_DIR / "ratings_processed.csv"

def split_pipe(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]

def count_pipe(value):
    return len(split_pipe(value))

def explode_pipe_counts(series):
    counts = Counter()
    for value in series.dropna():
        counts.update(split_pipe(value))
    return pd.DataFrame(counts.most_common(), columns=["value", "count"])

def parse_duration_minutes(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    if isinstance(value, (int, float)) and not pd.isna(value):
        return float(value) if float(value) > 0 else np.nan

    text = str(value).strip().lower()
    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        numeric = float(text)
        return numeric if numeric > 0 else np.nan

    hours = re.search(r"(\d+(?:\.\d+)?)\s*(?:hr|hour)", text)
    minutes = re.search(r"(\d+(?:\.\d+)?)\s*min", text)
    seconds = re.search(r"(\d+(?:\.\d+)?)\s*sec", text)
    total = 0.0
    if hours:
        total += float(hours.group(1)) * 60
    if minutes:
        total += float(minutes.group(1))
    if seconds:
        total += float(seconds.group(1)) / 60
    return total if total > 0 else np.nan

def infer_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [1, 2, 3]:
        return "winter"
    if month in [4, 5, 6]:
        return "spring"
    if month in [7, 8, 9]:
        return "summer"
    if month in [10, 11, 12]:
        return "fall"
    return np.nan

def add_bar_labels(ax, fmt="{:.0f}"):
    for patch in ax.patches:
        width = patch.get_width()
        ax.text(width, patch.get_y() + patch.get_height() / 2, " " + fmt.format(width), va="center", fontsize=9)

def save_current_plot(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")

df = pd.read_csv(ANIME_PATH)
summary_path = BUILD_DIR / "dataset_build_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
print(f"Anime rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(summary)

Anime rows: 14,965
Columns: 32


,mal_id,anidb_id,url,image_url,title,title_english,type,source,episodes,status,duration,total_watch_minutes,rating,score,scored_by,rank,popularity,members,favorites,synopsis,aired_year,aired_month,season,genres,explicit_tags,explicit_tag_weights,tags,tag_weights,demographics,studios,relations,recommendations
0,1,23.0,https://myanimelist.net/anime/1/Cowboy_Bebop,https://cdn.myanimelist.net/images/anime/4/19644l.jpg,Cowboy Bebop,Cowboy Bebop,TV,Original,26.0,Finished Airing,24.0,624.0,R - 17+ (violence & profanity),8.75,1064943,48.0,41,2063713,89939,"Crime is timeless. By the year 2071, humanity has expanded across the galaxy, filling the surface of other planets with settlements like those on Earth. The...",1998.0,4.0,spring,Action|Award Winning|Sci-Fi|Fantasy|Adventure|Suspense|Comedy|Romance,NaN,NaN,Adult Cast|Space|bounty hunter|Detective|reverse trap|cyborg|Martial Arts|Crossdressing|gunfights|swordplay|human enhancement|space travel|Gag Humor|medium ...,Adult Cast:0|Space:600|bounty hunter:600|Detective:400|reverse trap:200|cyborg:100|Martial Arts:200|Crossdressing:0|gunfights:400|swordplay:100|human enhanc...,Shoujo|Seinen,Sunrise,Side Story:5,205:319|6:206|889:132|400:129|20057:40|2251:29|4087:19|2025:21|918:14|13601:12|40052:11|467:10|1412:12|42310:9|36563:9|24439:9|30:9|790:6|40256:6|1575:6|474...
1,5,219.0,https://myanimelist.net/anime/5/Cowboy_Bebop__Tengoku_no_Tobira,https://cdn.myanimelist.net/images/anime/1439/93480l.jpg,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Movie,Original,1.0,Finished Airing,115.0,115.0,R - 17+ (violence & profanity),8.38,232607,241.0,659,412920,1793,"Another day, another bounty—such is the life of the often unlucky crew of the Bebop. However, this routine is interrupted when Faye, who is chasing a fairly...",2001.0,9.0,summer,Action|Sci-Fi|Comedy,NaN,NaN,Adult Cast|Space|bounty hunter|gunfights|plot continuity|Music|cyberpunk|nanomachines|neo-noir|air combat|terrorism,Adult Cast:0|Space:600|bounty hunter:600|gunfights:500|plot continuity:200|Music:0|cyberpunk:300|nanomachines:200|neo-noir:300|air combat:200|terrorism:200,Shoujo|Seinen,Bones,Parent Story:1,4106:3|21339:2|122:2|9135:1|23279:1|1796:1|393:1|522:1|570:1|1226:1
2,6,53.0,https://myanimelist.net/anime/6/Trigun,https://cdn.myanimelist.net/images/anime/1130/120002l.jpg,Trigun,Trigun,TV,Manga,26.0,Finished Airing,24.0,624.0,PG-13 - Teens 13 or older,8.22,402282,419.0,266,836983,17737,"Vash the Stampede is the man with a $$60,000,000,000 bounty on his head. The reason: he's a merciless villain who lays waste to all those that oppose him an...",1998.0,4.0,spring,Action|Adventure|Sci-Fi|Comedy|Romance,NaN,NaN,Adult Cast|cyborg|alien|gunfights|Mecha|Gag Humor|violence|humanoid alien|plot continuity|angst|Western|alcohol|law and order|tragedy|post-apocalyptic,Adult Cast:0|cyborg:200|alien:400|gunfights:600|Mecha:200|Gag Humor:400|violence:400|humanoid alien:500|plot continuity:300|angst:400|Western:400|alcohol:20...,Shounen,Madhouse,Side Story:4106|Alternative Version:52093,1:206|45:25|25:29|27:83|267:24|411:59|4981:9|24439:7|40256:6|297:57|68:19|400:21|790:5|205:5|1704:3|889:3|1470:3|2001:3|270:3|6030:10|482:2|268:2|777:2|2005...
3,7,96.0,https://myanimelist.net/anime/7/Witch_Hunter_Robin,https://cdn.myanimelist.net/images/anime/10/19969l.jpg,Witch Hunter Robin,Witch Hunter Robin,TV,Original,26.0,Finished Airing,25.0,650.0,PG-13 - Teens 13 or older,7.24,46350,3459.0,1997,129895,715,"Though hidden away from the general public, Witches—those with supernatural powers—have always existed in human societies. Neither numerous nor inherently e...",2002.0,7.0,summer,Action|Drama|Mystery|Supernatural|Sci-Fi|Fantasy|Horror|Suspense|Comedy|Romance,NaN,NaN,Detective|Military|painting|bishounen|magic|ghost|Crossdressing|gunfights|Super Power|human enhancement|contemporary fantasy|Parody|violence|plot continuity...,Detective:400|Military:100|painting:100|bishounen:100|magic:300|ghost:200|Crossdressing:100|gunfights:100|Super Power:200|huma

{'updated_at': '2026-06-12T19:37:23',
 'completed': False,
 'rows_collected': 14991,
 'failed_ids': 0,
 'invalid_type_ids': 6971,
 'permanent_http_skip_ids': 10,
 'anidb_cache_entries': 15725,
 'stats': {'processed': 30084,
  'added': 14973,
  'filtered': 8138,
  'failed': 38,
  'skipped_existing': 0,
  'skipped_invalid_type': 6955,
  'skipped_permanent_http': 0,
  'anidb_episode_fills': 106,
  'anidb_tag_fills': 64,
  'anidb_explicit_fills': 57,
  'anidb_demographic_fills': 73,
  'anidb_cache_hits': 12810,
  'anidb_live_requests': 106,
  'anidb_cooldowns': 0,
  'anidb_cooldown_skips': 0,
  'anidb_ban_resets': 0,
  'shoko_xml_cache_imported': 15638,
  'shoko_xml_cache_skipped_existing': 0,
  'shoko_xml_cache_parse_errors': 0,
  'recommendation_fills': 8255,
  'row_level_tag_cleanups': 0,
  'genre_values_promoted_from_tags': 26194,
  'duration_fills_from_anidb': 31}}

## Source Inventory

    Sources used:

    - Jikan API for MAL anime metadata. Jikan is an unofficial public API for MyAnimeList pages. It provides catalog fields such as title, type, score, status, dates, genres, themes, demographics, relations, recommendations, and external links.
    - AniDB HTTP API for AniDB metadata when needed, especially tags and episode counts for currently airing titles.
    - Shoko Anime HTTP XML cache as a cached AniDB XML source. This reduces live AniDB API calls and lowers ban risk.
    - Public MAL ID cache from `purarue/mal-id-cache` for candidate MAL IDs.
    - Kaggle user anime list dataset for ratings, stored after processing as `ratings_processed.csv`.

    Highest-risk source: AniDB live HTTP API, because it can ban clients when hit too aggressively. Mitigation: use Shoko cache first, use live calls only when needed, keep retry logs, and store deleted/missing AniDB IDs as permanent AniDB skips.

In [2]:
source_inventory = pd.DataFrame([
    {"source": "Jikan API", "role": "MAL catalog metadata", "format": "JSON", "local_artifact": "data/processed/anime_dataset.csv", "access": "public API, rate-limited"},
    {"source": "AniDB HTTP API", "role": "episode counts and tag metadata", "format": "XML", "local_artifact": "data/caches/anidb_metadata_cache.json", "access": "public API with strict usage limits"},
    {"source": "Shoko Anime_HTTP.zip", "role": "cached AniDB XML seed", "format": "ZIP of XML files", "local_artifact": "data/caches/anidb_metadata_cache.json", "access": "public cache"},
    {"source": "purarue MAL ID cache", "role": "candidate MAL IDs", "format": "JSON", "local_artifact": "data/raw/mal_candidate_ids.json", "access": "public GitHub data"},
    {"source": "Kaggle user anime list dataset", "role": "user-item ratings interaction layer", "format": "CSV", "local_artifact": "data/processed/ratings_processed.csv", "access": "Kaggle dataset terms"},
])
source_inventory

,source,role,format,local_artifact,access
0,Jikan API,MAL catalog metadata,JSON,data/processed/anime_dataset.csv,"public API, rate-limited"
1,AniDB HTTP API,episode counts and tag metadata,XML,data/caches/anidb_metadata_cache.json,public API with strict usage limits
2,Shoko Anime_HTTP.zip,cached AniDB XML seed,ZIP of XML files,data/caches/anidb_metadata_cache.json,public cache
3,purarue MAL ID cache,candidate MAL IDs,JSON,data/raw/mal_candidate_ids.json,public GitHub data
4,Kaggle user anime list dataset,user-item ratings interaction layer,CSV,data/processed/ratings_processed.csv,Kaggle dataset terms


## Schema Draft

    Main grain: one row per MAL anime entry in `anime_dataset`.

    Key fields:

    - `mal_id`: primary catalog key.
    - `anidb_id`: optional external key to AniDB.
    - `relations`: MAL relation graph edges stored as `relation_type:target_mal_id`.
    - `recommendations`: MAL recommendation edges stored as `target_mal_id:votes`.
    - `tags`, `tag_weights`, `explicit_tags`, `explicit_tag_weights`: feature fields from MAL and AniDB.
    - `ratings_processed.csv`: user-item interaction layer keyed by `animeID`, aligned to `mal_id`.

In [3]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "unique_values": [df[col].nunique(dropna=True) for col in df.columns],
})
schema

,column,dtype,non_null,missing,unique_values
0,mal_id,int64,14965,0,14965
1,anidb_id,float64,12819,2146,12309
2,url,object,14965,0,14965
3,image_url,object,14965,0,14954
4,title,object,14965,0,14965
5,title_english,object,8988,5977,8897
6,type,object,14965,0,6
7,source,object,14965,0,17
8,episodes,float64,14965,0,223
9,status,object,14965,0,2


## Data Dictionary Draft

    This draft gives meanings and quality notes for core fields. It is intentionally more than a column-name repeat.

In [4]:
data_dictionary = pd.DataFrame([
    {"field": "mal_id", "meaning": "MyAnimeList anime identifier", "type": "integer", "quality_note": "Primary key for catalog layer"},
    {"field": "anidb_id", "meaning": "AniDB anime identifier from MAL external links or fallback", "type": "integer/null", "quality_note": "May be missing or deleted in AniDB"},
    {"field": "type", "meaning": "MAL release type", "type": "category", "quality_note": "Allowed types include TV, Movie, OVA, ONA, Special, TV Special"},
    {"field": "score", "meaning": "MAL community score", "type": "float", "quality_note": "Rows without score are filtered out"},
    {"field": "episodes", "meaning": "Episode count", "type": "float/integer", "quality_note": "Airing shows may need AniDB refresh"},
    {"field": "duration", "meaning": "Episode duration in minutes", "type": "numeric", "quality_note": "Converted from MAL duration text during dataset improvements"},
    {"field": "total_watch_minutes", "meaning": "episodes multiplied by duration minutes", "type": "numeric", "quality_note": "Null when either episode count or duration is unavailable"},
    {"field": "genres", "meaning": "MAL genre list", "type": "pipe-delimited text", "quality_note": "Used for categorical features"},
    {"field": "tags", "meaning": "MAL themes plus AniDB non-explicit tags", "type": "pipe-delimited text", "quality_note": "AniDB weights preserve importance"},
    {"field": "explicit_tags", "meaning": "Adult/content-warning tags separated from normal tags", "type": "pipe-delimited text", "quality_note": "Missing only matters for explicit-rated rows"},
    {"field": "demographics", "meaning": "MAL or AniDB audience classification", "type": "pipe-delimited category", "quality_note": "AniDB can fill MAL empty demographics"},
    {"field": "relations", "meaning": "MAL relation edges", "type": "pipe-delimited graph edge list", "quality_note": "Supports relation graph"},
    {"field": "recommendations", "meaning": "MAL recommendation edges with votes", "type": "pipe-delimited weighted edge list", "quality_note": "Supports recommendation graph"},
])
data_dictionary

,field,meaning,type,quality_note
0,mal_id,MyAnimeList anime identifier,integer,Primary key for catalog layer
1,anidb_id,AniDB anime identifier from MAL external links or fallback,integer/null,May be missing or deleted in AniDB
2,type,MAL release type,category,"Allowed types include TV, Movie, OVA, ONA, Special, TV Special"
3,score,MAL community score,float,Rows without score are filtered out
4,episodes,Episode count,float/integer,Airing shows may need AniDB refresh
5,duration,Episode duration in minutes,numeric,Converted from MAL duration text during dataset improvements
6,total_watch_minutes,episodes multiplied by duration minutes,numeric,Null when either episode count or duration is unavailable
7,genres,MAL genre list,pipe-delimited text,Used for categorical features
8,tags,MAL themes plus AniDB non-explicit tags,pipe-delimited text,AniDB weights preserve importance
9,explicit_tags,Adult/content-warning tags separated from normal tags,pipe-delimited text,Missing only matters for explicit-rated rows


## Scale Analysis

    This table checks whether the dataset is non-trivial and whether missingness or sparsity will affect later modeling.

In [5]:
edge_cols = ["relations", "recommendations"]
scale = {
    "rows": len(df),
    "columns": df.shape[1],
    "unique_mal_id": df["mal_id"].nunique(),
    "unique_anidb_id": df["anidb_id"].nunique(dropna=True),
    "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "rows_with_tags": int(df["tags"].notna().sum()),
    "rows_with_recommendations": int(df["recommendations"].notna().sum()),
    "rows_with_relations": int(df["relations"].notna().sum()),
}
display(pd.DataFrame([scale]).T.rename(columns={0: "value"}))
missing = schema[["column", "missing"]].copy()
missing["missing_pct"] = missing["missing"] / len(df) * 100
display(missing.sort_values("missing_pct", ascending=False).head(15))

,value
rows,14965.00
columns,32.00
unique_mal_id,14965.00
unique_anidb_id,12309.00
memory_mb,31.07
rows_with_tags,12178.00
rows_with_recommendations,8388.00
rows_with_relations,7114.00


,column,missing,missing_pct
24,explicit_tags,12264,81.951220
25,explicit_tag_weights,12264,81.951220
30,relations,7851,52.462412
31,recommendations,6577,43.949215
5,title_english,5977,39.939860
28,demographics,5862,39.171400
26,tags,2787,18.623455
27,tag_weights,2787,18.623455
1,anidb_id,2146,14.340127
15,rank,1655,11.059138


## Processed Dataset V1 and Reproducibility

    Processed artifacts:

    - `data/processed/anime_dataset.csv`
    - `data/processed/anime_dataset.json`
    - `data/processed/ratings_processed.csv`
    - `data/caches/anidb_metadata_cache.json`
    - build logs and retry registries under `data/build/` and `logs/`

    Reproducible command path:

    - Dataset build notebook: run `notebooks/01_create_dataset.ipynb`.
    - Ratings notebook: run `notebooks/02_get_user_ratings.ipynb`.
    - Scripted feature build: run `python src/03_build_catalog_features.py`.

    The ingestion logs, checkpoint file, failed request registry, invalid type registry, and permanent skip registry provide evidence that the build is reproducible and auditable rather than manually assembled.

## Ethics and Access Note

    The catalog data comes from public MAL pages through Jikan and public AniDB metadata through AniDB/Shoko cache. The ratings data comes from a public Kaggle dataset and is used as anonymized user/item/rating interactions.

    The project does not scrape private profiles or bypass access controls. Personal-data risk is low but not zero because ratings are behavioral traces. We reduce risk by using processed user IDs only, not redistributing raw personal exports, and focusing on aggregate models rather than identifying users.

## Defense Preparation

    - Exact product question: anime discovery using content similarity, latent clusters, recommendation edges, and graph centrality.
    - Grain: one row per MAL anime; ratings are one row per user-anime interaction.
    - Highest-risk source: AniDB live API; mitigated with cache-first strategy and permanent skip rules.
    - Later layers: tags/text/numeric features, dimensionality reduction, K-means/DBSCAN clustering, recommendation graph, ratings matrix.
    - Limitations: MAL and AniDB disagree on specials; some AniDB IDs are deleted; recommendations are platform-generated and popularity-biased.